# Data processing pipeline — engineered version

This notebook preserves the original data-processing logic while reorganizing repeated operations into reusable functions.

The following are intentionally unchanged:

- the weekly Wednesday calendar;
- every source file, sheet, and source column;
- the distinct alignment rule used for each data source;
- forward filling, resampling, interpolation, differencing, and logarithmic transformations;
- the one-period covariate lag;
- the train/validation/test date boundaries;
- output filenames and the variable-availability table.


In [ ]:
import pandas as pd
import numpy as np
from scipy.stats import zscore


## 1. Central configuration

In [ ]:
# ---------------------------------------------------------------------
# Calendar and sample split
# ---------------------------------------------------------------------
CALENDAR_START = '2016-01-01'
CALENDAR_END = '2025-12-31'
WEEKLY_FREQUENCY = 'W-WED'

TRAIN_START = '2016-01-01'
VALID_START = '2022-01-01'
TEST_START = '2023-01-01'

# ---------------------------------------------------------------------
# Input files
# ---------------------------------------------------------------------
FILES = {
    'gcpu': 'GCPU.xlsx',
    'iqqh': 'clean_energy_etf_IQQH.csv',
    'icln': 'clean_energy_etf_ICLN.csv',
    'tan': 'clean_energy_etf_TAN.csv',
    'market_data': 'Libro1.xlsx',
    'global_clean_index': 'Global Clean Energy Transition Index.xls',
    'cpu_monthly': 'cpu_all_countries_monthly.csv',
    'gepu': 'Global_Economic_Policy_Uncertainty_EPU.xlsx',
    'itraxx': 'Libro2.xlsx',
    'rate_2y': 'rate_2yield.csv',
    'rate_10y': 'rate_10yield.csv',
    'vstoxx': 'v2tx.txt',
    'ttf': 'ICE Dutch TTF Natural Gas Futures Historical Data.csv',
    'brent': 'DCOILBRENTEU.csv',
}

# ---------------------------------------------------------------------
# Output files
# ---------------------------------------------------------------------
OUTPUT_FILES = {
    'train': 'train_dataset.csv',
    'valid': 'valid_dataset.csv',
    'test': 'test_dataset.csv',
}

# ---------------------------------------------------------------------
# Variables lagged by one weekly observation
# ---------------------------------------------------------------------
COVARIATE_COLS = [
    'GCPU_baseline',
    'l_t',
    'c_t',
    'itraxx_robustness',
    'CPU_EU_step',
    'CPU_EU_spline',
    'GEPU_current',
    'GEPU_ppp',
    'rate_10y',
    'Term_Spread',
    'log_VSTOXX',
    'TTF_return',
    'Brent_return',
]

# ---------------------------------------------------------------------
# Variables included in the availability table
# ---------------------------------------------------------------------
AVAILABILITY_VARS = [
    'GCPU_baseline',
    'y_IQQH_EUR',
    'y_ICLN',
    'y_TAN',
    'y_stoxx',
    'y_Global_Clean_Index',
    'CPU_EU_step',
    'CPU_EU_spline',
    'GEPU_current',
    'GEPU_ppp',
    'l_t',
    'c_t',
    'itraxx_robustness',
    'rate_10y',
    'Term_Spread',
    'log_VSTOXX',
    'TTF_return',
    'Brent_return',
    'COVID_dummy',
    'Energy_crisis_dummy',
]


## 2. Reusable low-level operations

In [ ]:
def create_weekly_calendar(start=CALENDAR_START, end=CALENDAR_END):
    """Create the original Wednesday-only master calendar."""
    return pd.DataFrame({
        'Date': pd.date_range(start=start, end=end, freq=WEEKLY_FREQUENCY)
    })


def merge_left(base, features, on):
    """Apply the left joins used throughout the original notebook."""
    return pd.merge(base, features, on=on, how='left')


def log_return_100(price):
    """Compute the original percentage-point log return: 100 * Δlog(price)."""
    return 100 * np.log(price).diff()


def get_return(file, col, name, skip=0):
    """
    Load a CSV price series and reproduce the original W-WED resampling rule.

    Important: this function deliberately uses the last available observation
    in each W-WED period. It does not replace the direct-Wednesday logic used
    for the STOXX and S&P index sources.
    """
    data = pd.read_csv(file, skiprows=skip)
    data.rename(columns={data.columns[0]: 'Date'}, inplace=True)
    data['Date'] = pd.to_datetime(data['Date'])
    data[col] = pd.to_numeric(data[col], errors='coerce')

    weekly = (
        data
        .set_index('Date')[[col]]
        .resample(WEEKLY_FREQUENCY)
        .last()
        .dropna(subset=[col])
        .reset_index()
    )
    weekly[name] = log_return_100(weekly[col])
    return weekly[['Date', name]]


def direct_wednesday_log_return(
    data,
    date_col,
    price_col,
    output_col,
    *,
    forward_fill=False,
    sort_dates=False,
    drop_missing_price=False,
    coerce_numeric=True,
):
    """
    Prepare sources for which the original notebook selected actual Wednesday
    rows rather than resampling to Wednesday.
    """
    prepared = data.copy()

    if date_col != 'Date':
        prepared.rename(columns={date_col: 'Date'}, inplace=True)

    if coerce_numeric:
        prepared[price_col] = pd.to_numeric(
            prepared[price_col],
            errors='coerce',
        )

    if forward_fill:
        prepared[price_col] = prepared[price_col].ffill()

    prepared = prepared[prepared['Date'].dt.weekday == 2]

    if sort_dates:
        prepared = prepared.sort_values('Date')

    if drop_missing_price:
        prepared = prepared.dropna(subset=[price_col])

    prepared = prepared.copy()
    prepared[output_col] = log_return_100(prepared[price_col])
    return prepared[['Date', output_col]]


## 3. Feature-block builders

In [ ]:
def add_gcpu_baseline(df):
    gcpu = pd.read_excel(FILES['gcpu'], sheet_name='GCPU_daily')
    gcpu['GCPU_baseline'] = pd.to_numeric(
        gcpu['GCPU(PPP-Adjusted GDP)'],
        errors='coerce',
    )
    gcpu_wed = gcpu[gcpu['date'].dt.weekday == 2][
        ['date', 'GCPU_baseline']
    ]

    return (
        pd.merge(
            df,
            gcpu_wed,
            left_on='Date',
            right_on='date',
            how='left',
        )
        .drop(columns=['date'])
    )


def add_clean_energy_and_placebo_returns(df):
    # Primary European clean-energy ETF
    df = merge_left(
        df,
        get_return(
            FILES['iqqh'],
            'Close',
            'y_IQQH_EUR',
            skip=[1, 2],
        ),
        on='Date',
    )

    # Global and solar robustness assets
    df = merge_left(
        df,
        get_return(
            FILES['icln'],
            'Close',
            'y_ICLN',
            skip=[1, 2],
        ),
        on='Date',
    )
    df = merge_left(
        df,
        get_return(
            FILES['tan'],
            'Close',
            'y_TAN',
            skip=[1, 2],
        ),
        on='Date',
    )

    # STOXX 600 Industrials placebo:
    # preserve the original forward-fill + actual-Wednesday rule.
    stoxx = pd.read_excel(
        FILES['market_data'],
        sheet_name='stoxx600 industrial',
    )
    stoxx_wed = direct_wednesday_log_return(
        stoxx,
        date_col='Timestamp',
        price_col='TRDPRC_1',
        output_col='y_stoxx',
        forward_fill=True,
        sort_dates=False,
        drop_missing_price=False,
        coerce_numeric=True,
    )
    df = merge_left(df, stoxx_wed, on='Date')

    # S&P Global Clean Energy Transition Index:
    # preserve the original header cleanup, forward fill, Wednesday selection,
    # sorting, and missing-price removal.
    idx_data = pd.read_excel(
        FILES['global_clean_index'],
        skiprows=6,
    )
    idx_data.columns = idx_data.columns.str.strip()
    idx_data['Date'] = pd.to_datetime(
        idx_data['Effective date'],
        errors='coerce',
    )

    price_col = 'S&P Global Clean Energy Transition Index (USD)'
    idx_wed = direct_wednesday_log_return(
        idx_data,
        date_col='Date',
        price_col=price_col,
        output_col='y_Global_Clean_Index',
        forward_fill=True,
        sort_dates=True,
        drop_missing_price=True,
        coerce_numeric=False,
    )
    return merge_left(df, idx_wed, on='Date')


In [ ]:
def prepare_cpu_monthly():
    cpu_monthly = pd.read_csv(FILES['cpu_monthly'])
    cpu_monthly.drop(columns='cit', inplace=True)
    cpu_monthly['year'] = cpu_monthly['year'].astype(int)
    cpu_monthly['month'] = cpu_monthly['month'].astype(int)
    cpu_monthly['ym'] = pd.to_datetime(
        cpu_monthly['year'].astype(str)
        + '-'
        + cpu_monthly['month'].astype(str)
        + '-01'
    ).dt.to_period('M')

    countries = [
        'CPU_DEU',
        'CPU_FRA',
        'CPU_ITA',
        'CPU_ESP',
        'CPU_IRL',
    ]

    for country_col in countries:
        cpu_monthly[country_col] = zscore(
            cpu_monthly[country_col],
            nan_policy='omit',
        )

    cpu_monthly['CPU_EU'] = cpu_monthly[countries].mean(axis=1)
    return cpu_monthly


def add_cpu_and_gepu_features(df):
    cpu_monthly = prepare_cpu_monthly()

    # Monthly step-function alignment
    df = df.copy()
    df['ym'] = df['Date'].dt.to_period('M')
    df = merge_left(
        df,
        cpu_monthly[['ym', 'CPU_EU']],
        on='ym',
    )
    df.rename(columns={'CPU_EU': 'CPU_EU_step'}, inplace=True)

    # Cubic-spline interpolation on a daily calendar, then merge to the
    # existing Wednesday master calendar. This reproduces the original logic.
    cpu_monthly['Interpolation_Date'] = (
        cpu_monthly['ym'].dt.to_timestamp()
    )
    full_daily = pd.DataFrame({
        'Date': pd.date_range(
            start=df['Date'].min(),
            end=df['Date'].max(),
            freq='D',
        )
    })
    full_daily = (
        pd.merge(
            full_daily,
            cpu_monthly[['Interpolation_Date', 'CPU_EU']],
            left_on='Date',
            right_on='Interpolation_Date',
            how='left',
        )
        .drop(columns=['Interpolation_Date'])
        .set_index('Date')
    )
    full_daily['CPU_EU'] = full_daily['CPU_EU'].interpolate(
        method='cubicspline',
        limit_area='inside',
    )
    full_daily = full_daily.reset_index()

    cpu_spline = full_daily[['Date', 'CPU_EU']].rename(
        columns={'CPU_EU': 'CPU_EU_spline'}
    )
    df = merge_left(df, cpu_spline, on='Date')

    # General economic policy uncertainty
    gepu = pd.read_excel(FILES['gepu'])
    gepu['ym'] = pd.to_datetime(
        gepu['Year'].astype(str)
        + '-'
        + gepu['Month'].astype(str)
        + '-01'
    ).dt.to_period('M')
    gepu['GEPU_current'] = pd.to_numeric(
        gepu['GEPU_current'],
        errors='coerce',
    )
    gepu['GEPU_ppp'] = pd.to_numeric(
        gepu['GEPU_ppp'],
        errors='coerce',
    )

    return merge_left(
        df,
        gepu[['ym', 'GEPU_current', 'GEPU_ppp']],
        on='ym',
    )


In [ ]:
def add_carbon_features(df):
    carbon = pd.read_excel(
        FILES['market_data'],
        sheet_name='FEUAc1',
    )
    carbon['EUA_Carbon'] = pd.to_numeric(
        carbon['SETTLE'],
        errors='coerce',
    )
    carbon.rename(columns={'Timestamp': 'Date'}, inplace=True)
    carbon['l_t'] = 100 * np.log(carbon['EUA_Carbon'])

    carbon_wed = carbon[carbon['Date'].dt.weekday == 2].copy()
    carbon_wed['c_t'] = carbon_wed['l_t'].diff()

    return merge_left(
        df,
        carbon_wed[['Date', 'l_t', 'c_t']],
        on='Date',
    )


def add_financial_and_energy_controls(df):
    # iTraxx robustness series
    itraxx = pd.read_excel(FILES['itraxx'])
    itraxx['itraxx'] = pd.to_numeric(
        itraxx['TRDPRC_1'],
        errors='coerce',
    )
    itraxx.rename(columns={'Timestamp': 'Date'}, inplace=True)
    itraxx_wed = itraxx[itraxx['Date'].dt.weekday == 2].copy()
    itraxx_wed['itraxx_robustness'] = itraxx_wed['itraxx'].diff()
    df = merge_left(
        df,
        itraxx_wed[['Date', 'itraxx_robustness']],
        on='Date',
    )

    # Euro-area rates and term spread
    rate_2y = pd.read_csv(
        FILES['rate_2y'],
        parse_dates=['observation_date'],
    )
    rate_10y = pd.read_csv(
        FILES['rate_10y'],
        parse_dates=['observation_date'],
    )
    rates = pd.merge(rate_2y, rate_10y, on='observation_date')
    rates['rate_2y'] = pd.to_numeric(
        rates['IR3TIB01EZM156N'],
        errors='coerce',
    )
    rates['rate_10y'] = pd.to_numeric(
        rates['IRLTLT01EZM156N'],
        errors='coerce',
    )
    rates['Term_Spread'] = (
        rates['rate_10y']
        - pd.to_numeric(rates['rate_2y'], errors='coerce')
    )
    rates['ym'] = rates['observation_date'].dt.to_period('M')
    df = merge_left(
        df,
        rates[['ym', 'rate_10y', 'Term_Spread']],
        on='ym',
    )

    # VSTOXX
    vstoxx = pd.read_csv(
        FILES['vstoxx'],
        sep=';',
        parse_dates=['Date'],
        dayfirst=True,
    )
    vstoxx['log_VSTOXX'] = 100 * np.log(
        pd.to_numeric(vstoxx['Indexvalue'], errors='coerce')
    )
    vstoxx_wed = vstoxx[vstoxx['Date'].dt.weekday == 2][
        ['Date', 'log_VSTOXX']
    ]
    df = merge_left(df, vstoxx_wed, on='Date')

    # TTF gas and Brent oil returns
    df = merge_left(
        df,
        get_return(
            FILES['ttf'],
            'Price',
            'TTF_return',
        ),
        on='Date',
    )
    df = merge_left(
        df,
        get_return(
            FILES['brent'],
            'DCOILBRENTEU',
            'Brent_return',
        ),
        on='Date',
    )

    df.drop(columns=['ym'], inplace=True)
    return df


In [ ]:
def add_event_dummies(df):
    df = df.copy()

    df['COVID_dummy'] = (
        (df['Date'] >= '2020-02-01')
        & (df['Date'] <= '2023-06-30')
    ).astype(int)

    df['Energy_crisis_dummy'] = (
        (df['Date'] >= '2021-01-01')
        & (df['Date'] <= '2022-12-31')
    ).astype(int)

    return df


def lag_covariates(df, columns=COVARIATE_COLS):
    df = df.copy()
    df.loc[:, columns] = df.loc[:, columns].shift(1)
    return df


def split_dataset(df):
    # Preserve the original removal of the first row after lagging.
    df = df.drop(df.index[0])

    train = df[
        (df['Date'] >= TRAIN_START)
        & (df['Date'] < VALID_START)
    ].copy()

    valid = df[
        (df['Date'] >= VALID_START)
        & (df['Date'] < TEST_START)
    ].copy()

    test = df[
        df['Date'] >= TEST_START
    ].copy()

    return df, train, valid, test


def save_dataset_splits(train, valid, test):
    datasets = {
        'train': train,
        'valid': valid,
        'test': test,
    }

    for split_name, split_df in datasets.items():
        split_df.to_csv(
            OUTPUT_FILES[split_name],
            index=False,
            encoding='utf-8-sig',
        )


def make_availability_table(df, variables=AVAILABILITY_VARS):
    rows = []

    for col in variables:
        observed = df[['Date', col]].dropna(subset=[col])

        if len(observed) == 0:
            start_date = None
            end_date = None
        else:
            start_date = observed['Date'].min().date()
            end_date = observed['Date'].max().date()

        rows.append({
            'Variable': col,
            'Start date': start_date,
            'End date': end_date,
            'Missing %': round(df[col].isna().mean() * 100, 2),
        })

    return pd.DataFrame(rows)


## 4. End-to-end orchestration

In [ ]:
def build_processed_dataset():
    """Run the original processing sequence in one explicit pipeline."""
    df = create_weekly_calendar()
    df = add_gcpu_baseline(df)
    df = add_clean_energy_and_placebo_returns(df)
    df = add_cpu_and_gepu_features(df)
    df = add_carbon_features(df)
    df = add_financial_and_energy_controls(df)
    df = add_event_dummies(df)
    df = lag_covariates(df)
    return df


## 5. Run the pipeline and export results

In [ ]:
df_clean = build_processed_dataset()
df_clean, train, valid, test = split_dataset(df_clean)

save_dataset_splits(train, valid, test)

df_clean.head()


In [ ]:
df_table = make_availability_table(df_clean)

display(df_table.head())
print(df_table.to_latex(index=False))
